In [ ]:
#!/usr/bin/env python
"""
Train a Logistic Regression model for healthcare provider fraud detection.

This script reproduces the feature engineering from the Random Forest notebook,
but swaps the RF model for a Logistic Regression model.
"""

import os
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
# import joblib  # uncomment if you want to save the trained model


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

# Folder where your CSV files live – CHANGE THIS to your own path
DATA_DIR = "./data"

# File names – CHANGE these if your filenames differ
MAIN_CLAIM_FILE = "Anonymized_Train-1542865627584.csv"
BENEFICIARY_FILE = "Anonymized_Train_Beneficiarydata.csv"
INPATIENT_FILE = "Anonymized_Train_Inpatientdata.csv"
OUTPATIENT_FILE = "Anonymized_Train_Outpatientdata.csv"


# ---------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------

def load_data():
    """Load raw CSVs into DataFrames."""
    main_claim_df = pd.read_csv(os.path.join(DATA_DIR, MAIN_CLAIM_FILE))
    beneficiary_df = pd.read_csv(os.path.join(DATA_DIR, BENEFICIARY_FILE))
    inpatient_df = pd.read_csv(os.path.join(DATA_DIR, INPATIENT_FILE))
    outpatient_df = pd.read_csv(os.path.join(DATA_DIR, OUTPATIENT_FILE))

    return main_claim_df, beneficiary_df, inpatient_df, outpatient_df


# ---------------------------------------------------------------------
# Feature engineering (mirrors your notebook logic)
# ---------------------------------------------------------------------

def build_feature_table(main_claim_df, beneficiary_df, inpatient_df, outpatient_df):
    """
    Reproduce the final_df / X / y construction from the notebook:
    - combine inpatient + outpatient claims
    - merge with beneficiary info
    - date & LOS features
    - aggregate by Provider
    - encode categorical variables
    """

    # Combine inpatient and outpatient claims
    claims_df = pd.concat([inpatient_df, outpatient_df], ignore_index=True)

    # Merge with beneficiary info
    merged_claims = claims_df.merge(beneficiary_df, on='BeneID', how='left')

    # Convert date columns and compute length of stay (LOS)
    merged_claims['ClaimStartDt'] = pd.to_datetime(
        merged_claims['ClaimStartDt'], errors='coerce'
    )
    merged_claims['ClaimEndDt'] = pd.to_datetime(
        merged_claims['ClaimEndDt'], errors='coerce'
    )
    merged_claims['LOS'] = (
        merged_claims['ClaimEndDt'] - merged_claims['ClaimStartDt']
    ).dt.days

    # Convert monetary columns to numeric
    merged_claims['InscClaimAmtReimbursed'] = pd.to_numeric(
        merged_claims['InscClaimAmtReimbursed'], errors='coerce'
    )
    merged_claims['DeductibleAmtPaid'] = pd.to_numeric(
        merged_claims['DeductibleAmtPaid'], errors='coerce'
    )

    # Aggregate features by Provider (as in your notebook)
    agg_df = merged_claims.groupby('Provider').agg({
        'ClaimID': 'count',
        'InscClaimAmtReimbursed': 'sum',
        'DeductibleAmtPaid': 'sum',
        'LOS': 'mean',
        'Gender': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        'Race': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        'ChronicCond_Diabetes': 'mean',
        'ChronicCond_Heartfailure': 'mean',
        'ChronicCond_Depression': 'mean',
        'ChronicCond_IschemicHeart': 'mean',
        'ChronicCond_stroke': 'mean',
        'IPAnnualReimbursementAmt': 'mean',
        'OPAnnualReimbursementAmt': 'mean',
        'IPAnnualDeductibleAmt': 'mean',
        'OPAnnualDeductibleAmt': 'mean'
    }).reset_index()

    # Rename columns for clarity (same as notebook)
    agg_df.columns = [
        'Provider', 'TotalClaims', 'TotalReimbursed', 'TotalDeductible', 'Avg_LOS',
        'MostCommonGender', 'MostCommonRace',
        'Avg_Diabetes', 'Avg_HeartFailure', 'Avg_Depression',
        'Avg_IschemicHeart', 'Avg_Stroke',
        'Avg_IP_Reimbursed', 'Avg_OP_Reimbursed',
        'Avg_IP_Deductible', 'Avg_OP_Deductible'
    ]

    # Merge with fraud label from main_claim_df
    final_df = agg_df.merge(
        main_claim_df[['Provider', 'PotentialFraud']],
        on='Provider',
        how='inner'
    )

    # Encode categorical features and map labels
    le_gender = LabelEncoder()
    le_race = LabelEncoder()

    final_df['MostCommonGender'] = le_gender.fit_transform(
        final_df['MostCommonGender'].astype(str)
    )
    final_df['MostCommonRace'] = le_race.fit_transform(
        final_df['MostCommonRace'].astype(str)
    )
    final_df['FraudLabel'] = final_df['PotentialFraud'].map({'Yes': 1, 'No': 0})

    # Build X and y as in your notebook
    X = final_df.drop(columns=['Provider', 'PotentialFraud', 'FraudLabel'])
    y = final_df['FraudLabel']

    return X, y, final_df


# ---------------------------------------------------------------------
# Model building – logistic regression instead of random forest
# ---------------------------------------------------------------------

def build_logistic_pipeline():
    """Create the scaler + logistic regression pipeline."""
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(
            penalty='l2',
            C=1.0,
            class_weight='balanced',  # analogous to RF class_weight
            max_iter=1000,
            solver='lbfgs'
        ))
    ])
    return pipeline


def evaluate_model(pipeline, X, y):
    """Run CV and hold-out evaluation."""
    # Stratified K-fold CV (ROC AUC)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    print("Performing 5-Fold Cross-Validation (ROC AUC)...")
    cv_scores = cross_val_score(
        pipeline, X, y, cv=cv, scoring='roc_auc'
    )
    print(f"CV ROC AUC scores: {cv_scores}")
    print(f"Mean ROC AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}\n")

    # Train-test split for a simple hold-out evaluation
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    print("Classification report (Logistic Regression):")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion matrix:")
    print(cm)

    auc = roc_auc_score(y_test, y_proba)
    print(f"Hold-out ROC AUC: {auc:.4f}")

    return pipeline


# ---------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------

def main():
    print("Loading data...")
    main_claim_df, beneficiary_df, inpatient_df, outpatient_df = load_data()

    print("Building features...")
    X, y, final_df = build_feature_table(
        main_claim_df, beneficiary_df, inpatient_df, outpatient_df
    )

    print(f"Feature matrix shape: {X.shape}")
    print(f"Label distribution:\n{y.value_counts()}\n")

    print("Building logistic regression pipeline...")
    pipeline = build_logistic_pipeline()

    print("Training and evaluating model...")
    pipeline = evaluate_model(pipeline, X, y)

    # Optionally save the trained pipeline
    # joblib.dump(pipeline, "logistic_fraud_model.joblib", compress=3)
    # print("\nModel saved to logistic_fraud_model.joblib")


if __name__ == "__main__":
    main()
